# 06 · Dense event autoencoder + XGBoost
Sections 1–3 verify the frozen encoder, compressor and five fold ensembles. Sections 4–5 retrain the compressor and trees. Full compressor training temporarily needs 15 GiB for residuals.

## 1. Load the frozen teacher and the dense autoencoder

In [ ]:
from pathlib import Path
import json
import math

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'environment/shared.ipynb').exists())
def load_cells(path):
    for cell in json.loads(path.read_text())['cells']:
        if cell['cell_type'] == 'code':
            exec(''.join(cell['source']), globals())

load_cells(ROOT / 'environment/shared.ipynb')
import torch
train = pd.read_csv(ROOT / 'data/raw/train.csv', dtype={'event': str, 'user': str})
test = pd.read_csv(ROOT / 'data/raw/test.csv', dtype={'event': str, 'user': str})
samples = pd.concat([train.drop(columns=['interested', 'not_interested']), test], ignore_index=True)
folds = pd.read_csv(ROOT / 'data/folds.csv').fold.to_numpy()
base = pd.read_pickle(ROOT / '02_xgboost_baseline/reference/features.pkl')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.set_num_threads(4)
HERE = ROOT / '06_dense_event_autoencoder'
reference = HERE / 'reference'
output = HERE / 'results'
output.mkdir(exist_ok=True)
load_cells(ROOT / '03_masked_event_transformer/event_encoder.ipynb')
event_ref = ROOT / '03_masked_event_transformer/reference'
meta = json.loads((event_ref / 'encoder.json').read_text())
teacher = build_model(meta['cards']).to(device)
teacher.load_state_dict(torch.load(event_ref / 'encoder.pt', map_location=device))
teacher.eval()
event_ids = pd.Index(pd.read_csv(event_ref / 'candidate_residual_events.csv', dtype=str).event)
row_index = event_ids.get_indexer(samples.event)
class DenseAutoencoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = torch.nn.Sequential(torch.nn.Linear(1280, 64), torch.nn.SiLU())
        self.decoder = torch.nn.Linear(64, 1280)
    def forward(self, residual):
        latent = self.encoder(residual)
        return self.decoder(latent), latent

compressor = DenseAutoencoder().to(device)
compressor.load_state_dict(torch.load(reference / 'autoencoder.pt', map_location=device))
compressor.eval()

## 2. Check the 64 values for each candidate

In [ ]:
@torch.inference_mode()
def compress(model, residuals):
    chunks = []
    for start in range(0, len(residuals), 4096):
        block = torch.as_tensor(np.array(residuals[start:start + 4096]), device=device)
        chunks.append(model.encoder(block).cpu().numpy())
    return np.concatenate(chunks)

def add_latents(base, latent):
    columns = [f'dense64_{i:02d}' for i in range(64)]
    return pd.concat([base, pd.DataFrame(latent, columns=columns, index=base.index)], axis=1)

saved = np.load(reference / 'candidate_latents.npy')
residuals = np.concatenate(list(residual_batches(teacher, np.load(event_ref / 'candidate_codes.npy'))))
restored = compress(compressor, residuals)[row_index]
print('Maximum latent difference', float(np.abs(restored - saved).max()))
x = add_latents(base, saved)

## 3. Score the five saved folds
Evaluate the original MPS inputs and then the coordinates regenerated on this device. Small floating-point differences can move values across a tree split.

In [ ]:
scores = evaluate_saved(x, train, folds, reference, [42, 2026], baseline=False)
expected = pd.read_csv(reference / 'oof.csv').probability
print('OOF MAP@200', map_users(train, scores).ap200.mean())
print('Maximum probability difference', float(np.abs(scores - expected).max()))
regenerated = add_latents(base, restored)
end_to_end = evaluate_saved(regenerated, train, folds, reference, [42, 2026], baseline=False)
print('MAP@200 with regenerated neural inputs', map_users(train, end_to_end).ap200.mean())


## 4. Train the compressor on all frozen residuals
Use a temporary memory-mapped array. The training target is the original 1,280-value vector.

In [ ]:
CONFIG = dict(input_dim=1280, latent_dim=64, activation='SiLU', tied_weights=False, epochs=20, batch_size=4096, block_rows=65536, seed=42, lr=0.001, lr_final=0.00001, weight_decay=0.0, loss='raw residual elementwise MSE', dtype='float32', parameters=165184)
def fit_compressor(residuals):
    torch.manual_seed(42)
    model = DenseAutoencoder().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=0)
    starts = np.arange(0, len(residuals), CONFIG['block_rows'])
    steps_per_epoch = sum(math.ceil(min(CONFIG['block_rows'], len(residuals) - int(start)) / CONFIG['batch_size']) for start in starts)
    total_steps = steps_per_epoch * CONFIG['epochs']
    rng = np.random.default_rng(42)
    step = 0
    for epoch in range(1, CONFIG['epochs'] + 1):
        model.train()
        squared_error, elements, seen = 0.0, 0, 0
        for start in rng.permutation(starts):
            block = torch.from_numpy(np.array(residuals[start:start + CONFIG['block_rows']], dtype=np.float32)).to(device)
            order = rng.permutation(len(block))
            for offset in range(0, len(block), CONFIG['batch_size']):
                index = torch.as_tensor(order[offset:offset + CONFIG['batch_size']], dtype=torch.long, device=device)
                target = block[index]
                lr = CONFIG['lr_final'] + 0.5 * (CONFIG['lr'] - CONFIG['lr_final']) * (1 + math.cos(math.pi * step / max(total_steps - 1, 1)))
                optimizer.param_groups[0]['lr'] = lr
                optimizer.zero_grad(set_to_none=True)
                reconstructed, latent = model(target)
                loss = torch.nn.functional.mse_loss(reconstructed, target)
                loss.backward()
                optimizer.step()
                squared_error += float(loss.detach().item()) * target.numel()
                elements += target.numel()
                seen += len(target)
                step += 1
            del block
        print(epoch, squared_error / elements)
    return model.eval()

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory(dir=output) as scratch:
    codes, catalog_ids = encode_catalog(ROOT / 'data/raw/events.csv.gz', meta, Path(scratch))
    raw = np.lib.format.open_memmap(Path(scratch) / 'residuals.npy', mode='w+', dtype='float32', shape=(len(codes), 1280))
    position = 0
    for batch in residual_batches(teacher, codes):
        raw[position:position + len(batch)] = batch
        position += len(batch)
    fitted = fit_compressor(raw)
    torch.save(fitted.state_dict(), output / 'autoencoder.pt')
    unique_latents = compress(fitted, raw[catalog_ids.get_indexer(event_ids.astype('uint64'))])
    del raw, codes

## 5. Rebuild original features and retrain five fold ensembles

In [ ]:
original, _ = features(ROOT / 'data/raw', samples)
x = add_latents(original, unique_latents[row_index])
parameters = json.loads((HERE / 'parameters.json').read_text())
print(train_folds(x, train, test, folds, parameters, [42, 2026], output))